<a href="https://colab.research.google.com/github/sabarik7180/DE_basics/blob/main/dea_brazilian_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
print(os.getcwd())

/content


In [3]:
# Install dependencies as needed:
!pip install kagglehub[pandas-datasets]

Now, let's correct the `file_path` in the next cell to specify a `.csv` file from the dataset.

In [6]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "olist_customers_dataset.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "olistbr/brazilian-ecommerce",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

/tmp/ipykernel_1744/1098941792.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
First 5 records:                         customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  


In [10]:
import os
dataset_path = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
dataset_files = os.listdir(dataset_path)
for file in dataset_files:
    print(file)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
olist_customers_dataset.csv
olist_sellers_dataset.csv
olist_order_reviews_dataset.csv
olist_order_items_dataset.csv
olist_products_dataset.csv
olist_geolocation_dataset.csv
product_category_name_translation.csv
olist_orders_dataset.csv
olist_order_payments_dataset.csv


In [11]:
!ls -lrt -h /kaggle/input/brazilian-ecommerce/

total 121M
-rw-r--r-- 1 1000 1000 2.6K Feb 24 15:00 product_category_name_translation.csv
-rw-r--r-- 1 1000 1000 171K Feb 24 15:00 olist_sellers_dataset.csv
-rw-r--r-- 1 1000 1000 2.3M Feb 24 15:00 olist_products_dataset.csv
-rw-r--r-- 1 1000 1000 5.6M Feb 24 15:00 olist_order_payments_dataset.csv
-rw-r--r-- 1 1000 1000 8.7M Feb 24 15:00 olist_customers_dataset.csv
-rw-r--r-- 1 1000 1000  14M Feb 24 15:00 olist_order_reviews_dataset.csv
-rw-r--r-- 1 1000 1000  17M Feb 24 15:00 olist_orders_dataset.csv
-rw-r--r-- 1 1000 1000  15M Feb 24 15:00 olist_order_items_dataset.csv
-rw-r--r-- 1 1000 1000  59M Feb 24 15:00 olist_geolocation_dataset.csv


In [12]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('brazilian Dataset dea').getOrCreate()

customer_df = spark.read.csv('/kaggle/input/brazilian-ecommerce/olist_customers_dataset.csv', header=True, inferSchema=True)
customer_df.printSchema()
customer_df.show(10)


root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83

In [13]:
from pyspark.sql import SparkSession
import os

# Ensure SparkSession is initialized
spark = SparkSession.builder.appName('brazilian Dataset dea').getOrCreate()

# Assuming dataset_path and dataset_files are available from previous cells
# If not, uncomment and run these lines:
# import kagglehub
# dataset_path = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
# dataset_files = os.listdir(dataset_path)

dataframes = {}

# Exclude the customer dataset as it was already loaded in a previous cell
loaded_files = ['olist_customers_dataset.csv']

for file_name in dataset_files:
    if file_name.endswith('.csv') and file_name not in loaded_files:
        # Create a clean DataFrame name
        df_name = file_name.replace('olist_', '').replace('_dataset.csv', '').replace('.csv', '')

        full_file_path = os.path.join(dataset_path, file_name)
        print(f"\n--- Loading {file_name} as {df_name}_df ---")
        try:
            df = spark.read.csv(full_file_path, header=True, inferSchema=True)
            df.printSchema()
            df.show(10)
            dataframes[f'{df_name}_df'] = df
        except Exception as e:
            print(f"Error loading {file_name}: {e}")

# You can now access individual dataframes, e.g., dataframes['sellers_df']


--- Loading olist_sellers_dataset.csv as sellers_df ---
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

+--------------------+----------------------+-----------------+------------+
|           seller_id|seller_zip_code_prefix|      seller_city|seller_state|
+--------------------+----------------------+-----------------+------------+
|3442f8959a84dea7e...|                 13023|         campinas|          SP|
|d1b65fc7debc3361e...|                 13844|       mogi guacu|          SP|
|ce3ad9de960102d06...|                 20031|   rio de janeiro|          RJ|
|c0f3eea2e14555b6f...|                  4195|        sao paulo|          SP|
|51a04a8a6bdcb23de...|                 12914|braganca paulista|          SP|
|c240c4061717ac180...|                 20920|   rio de janeiro|          RJ|
|e49c26c3edfa46d22...|                 55325|           breja

```dbml
// Use DBML to define your database structure
// Docs: https://dbml.dbdiagram.io/docs

Project "Brazilian E-commerce Dataset" {
  database_type: 'PostgreSQL'
  Note: 'Schema for the Olist Brazilian E-commerce Public Dataset'
}

Table customer {
  customer_id varchar [pk]
  customer_unique_id varchar
  customer_zip_code_prefix integer
  customer_city varchar
  customer_state varchar
}

Table seller {
  seller_id varchar [pk]
  seller_zip_code_prefix integer
  seller_city varchar
  seller_state varchar
}

Table product {
  product_id varchar [pk]
  product_category_name varchar [note: 'FK to product_category_name_translation']
  product_name_lenght integer
  product_description_lenght integer
  product_photos_qty integer
  product_weight_g integer
  product_length_cm integer
  product_height_cm integer
  product_width_cm integer
}

Table product_category_name_translation {
  product_category_name varchar [pk, note: 'Translated category name']
  product_category_name_english varchar
}

Table "order" {
  order_id varchar [pk]
  customer_id varchar [not null, note: 'FK to customer']
  order_status varchar
  order_purchase_timestamp varchar [note: 'Should be timestamp']
  order_approved_at varchar [note: 'Should be timestamp']
  order_delivered_carrier_date varchar [note: 'Should be timestamp']
  order_delivered_customer_date varchar [note: 'Should be timestamp']
  order_estimated_delivery_date varchar [note: 'Should be timestamp']
}

Table order_item {
  order_id varchar [pk, note: 'Part of composite PK, FK to order']
  order_item_id integer [pk, note: 'Part of composite PK']
  product_id varchar [not null, note: 'FK to product']
  seller_id varchar [not null, note: 'FK to seller']
  shipping_limit_date varchar [note: 'Should be timestamp']
  price float
  freight_value float
}

Table order_payment {
  order_id varchar [pk, note: 'Part of composite PK, FK to order']
  payment_sequential integer [pk, note: 'Part of composite PK']
  payment_type varchar
  payment_installments integer
  payment_value float
}

Table order_review {
  review_id varchar [pk]
  order_id varchar [not null, note: 'FK to order']
  review_score varchar [note: 'Spark schema inferred string, often integer']
  review_comment_title varchar
  review_comment_message text
  review_creation_date varchar [note: 'Should be timestamp']
  review_answer_timestamp varchar [note: 'Should be timestamp']
}

Table geolocation {
  geolocation_zip_code_prefix integer [note: 'Can be joined with customer/seller zip codes']
  geolocation_lat float
  geolocation_lng float
  geolocation_city varchar
  geolocation_state varchar
}

// --- Relationships ---
Ref: "order".customer_id > customer.customer_id
Ref: order_item.order_id > "order".order_id
Ref: order_item.product_id > product.product_id
Ref: order_item.seller_id > seller.seller_id
Ref: order_payment.order_id > "order".order_id
Ref: order_review.order_id > "order".order_id
Ref: product.product_category_name > product_category_name_translation.product_category_name

// Implicit joins (not explicit FKs but conceptual links)
// customer.customer_zip_code_prefix <-> geolocation.geolocation_zip_code_prefix
// seller.seller_zip_code_prefix <-> geolocation.geolocation_zip_code_prefix
```